In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional

coverage = pd.read_csv("coverage.tsv", sep="\t")

In [ ]:
def plot_chr_read_fraction(
    df: pd.DataFrame, out_dir: str = "plots_chr_fraction"
) -> None:
    """
    Plot read percentage per chromosome for each sample.

    Parameters
    df : pd.DataFrame
        DataFrame with 'chr', 'start' columns and sample columns with read counts.
    out_dir : str, default="plots_chr_fraction"
        Output directory for saved plots.

    Returns
    None
        Saves plots to out_dir and closes them.
    """
    out = Path(out_dir)
    out.mkdir(exist_ok=True)

    sample_cols = [c for c in df.columns if c not in ["chr", "start"]]

    df = df.copy()
    df["chr"] = df["chr"].astype(str)

    for sample in sample_cols:

        chr_sum = df.groupby("chr")[sample].sum()

        total = chr_sum.sum()

        ratio = chr_sum / total

        chrom_order = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

        ratio = ratio.loc[[c for c in chrom_order if c in ratio.index]]

        ratio_percent = ratio * 100

        plt.figure(figsize=(10, 5))

        plt.plot(ratio_percent.index, ratio_percent.values, marker="o")

        plt.xlabel("Chromosome")
        plt.ylabel("Reads (%)")
        plt.title(f"{sample} read distribution")

        plt.xticks(rotation=90)
        plt.tight_layout()

        plt.savefig(out / f"{sample}_chr_fraction.png", dpi=150)
        plt.close()

In [3]:
plot_chr_read_fraction(coverage)

In [ ]:
def plot_chr_fraction_multi(
    df: pd.DataFrame,
    samples: Optional[List[str]] = None,
    out_file: str = "chr_fraction_common.png",
) -> None:
    """
    Plot read percentage per chromosome for multiple samples on same figure.

    Parameters
    df : pd.DataFrame
        DataFrame with 'chr', 'start' columns and sample columns with read counts.
    samples : list of str, optional
        List of sample columns to plot. If None, uses all non-meta columns.
    out_file : str, default="chr_fraction_common.png"
        Output file path for the saved plot.

    Returns
    None
        Saves plot to file and closes it.
    """
    all_samples = [c for c in df.columns if c not in ["chr", "start"]]

    if samples is None:
        samples = all_samples
    else:
        samples = [s for s in samples if s in all_samples]

    df = df.copy()
    df["chr"] = df["chr"].astype(str)

    chrom_order = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

    plt.figure(figsize=(12, 5))

    for sample in samples:

        chr_sum = df.groupby("chr")[sample].sum()
        total = chr_sum.sum()

        ratio = (chr_sum / total) * 100
        ratio = ratio.loc[[c for c in chrom_order if c in ratio.index]]

        plt.plot(
            ratio.index,
            ratio.values,
            marker="o",
            linewidth=1,
            alpha=0.6,
            label=sample.split("_")[0],
        )

    plt.xlabel("Chromosome")
    plt.ylabel("Reads (%)")
    plt.title("Read distribution across chromosomes")

    plt.xticks(rotation=90)

    plt.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=len(samples), fontsize=8
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig(out_file, dpi=150, bbox_inches="tight")
    plt.close()

In [5]:
plot_chr_fraction_multi(coverage)